# Lecture 12: Normalizing Flows


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm as scipy_norm
from scipy.special import expit as sigmoid
from sklearn.datasets import make_moons

np.random.seed(42)
print("Libraries loaded.")


## 1. Core Idea: Distribution Transformation (Ch. 16.1)

The goal of a Normalizing Flow is to transform a **simple distribution** (e.g. Gaussian) into a complex one via an **invertible** network.

**Change-of-variables formula (1D):**

$$p_x(x) = p_z(z) \\cdot \\left|\\frac{\\partial f^{-1}[x]}{\\partial x}\\right| = p_z(z) \\cdot \\left|\\frac{\\partial f[z]}{\\partial z}\\right|^{-1}$$

- $z \\sim p_z$ : Simple base distribution
- $x = f(z)$ : Learned invertible transformation
- $\\left|\\frac{\\partial f}{\\partial z}\\right|$ : Jacobian determinant — measures volume change


In [ ]:
# ─── Visualize 1D distribution transformation ───
z_range = np.linspace(-4, 4, 500)
p_z = scipy_norm.pdf(z_range)     # standard Gaussian (base)

# Transformation: x = f(z) = z^3 / 6 + z   (monotone, invertible)
def f_transform(z):
    return z**3 / 6 + z

def f_inv_approx(x, z_init=None):
    # Numerical inverse: Newton iteration
    z = np.zeros_like(x) if z_init is None else z_init.copy()
    for _ in range(50):
        fx = f_transform(z)
        dfz = z**2 / 2 + 1   # f'(z)
        z = z - (fx - x) / dfz
    return z

def jacobian_f(z):
    return np.abs(z**2 / 2 + 1)   # |df/dz|

x_range = f_transform(z_range)
# p_x(x) = p_z(f_inv(x)) * |d f_inv / dx| = p_z(z) / |df/dz|
p_x = p_z / jacobian_f(z_range)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].fill_between(z_range, p_z, alpha=0.5, color='steelblue')
axes[0].plot(z_range, p_z, color='steelblue', lw=2)
axes[0].set_title("Base Distribution $p_z(z)$\nStandard Gaussian", fontsize=11)
axes[0].set_xlabel("z")

axes[1].plot(z_range, x_range, color='darkorange', lw=2.5)
axes[1].axhline(0, color='gray', lw=0.5); axes[1].axvline(0, color='gray', lw=0.5)
axes[1].set_title("Transformation $x = f(z) = z^3/6 + z$\n(monotone = invertible)", fontsize=11)
axes[1].set_xlabel("z"); axes[1].set_ylabel("x")

# Density in x space
axes[2].fill_between(x_range, p_x, alpha=0.5, color='crimson')
axes[2].plot(x_range, p_x, color='crimson', lw=2)
axes[2].set_title("Transformed Distribution $p_x(x)$\nAsymmetric, heavy tail", fontsize=11)
axes[2].set_xlabel("x")
axes[2].set_xlim(-6, 6)

plt.suptitle("Normalizing Flow: Simple → Complex Distribution", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig("fig_nb12_flow_1d.png", dpi=100, bbox_inches='tight')
plt.show()

# Sanity check (integral should be ~1)
dx = np.diff(x_range)
approx_integral = np.sum(p_x[:-1] * np.abs(dx))
print(f"p_x integral (approx): {approx_integral:.4f}  (should be 1.0)")


## 2. Log-Likelihood and Training (Ch. 16.1.3)

Log-likelihood for a normalizing flow:

$$\\log p_x(x_i) = \\log p_z(z_i) - \\log\\left|\\frac{\\partial f[z_i]}{\\partial z_i}\\right|$$

$$\\hat\\phi = \\arg\\min_\\phi \\sum_{i=1}^N \\left[ -\\log p_z(z_i) + \\log\\left|\\frac{\\partial f[z_i,\\phi]}{\\partial z_i}\\right| \\right]$$

**Balance between two terms:**
- $-\\log p_z(z)$: How well $z$ fits the base distribution
- $\\log |J|$: Volume change correction (large Jacobian = large expansion)


In [ ]:
# Visualize log-likelihood over a parameter (scale)
np.random.seed(5)
x_data = np.concatenate([
    np.random.normal(-2, 0.5, 200),
    np.random.normal(2, 0.5, 200)
])  # bimodal data

def affine_flow(z, scale, shift):
    return scale * z + shift

def log_likelihood_affine(x_data, scale, shift):
    z = (x_data - shift) / scale          # inverse transform
    log_pz = scipy_norm.logpdf(z)          # log p_z(z)
    log_jacob = -np.log(np.abs(scale))     # log |dz/dx| = -log|scale|
    return np.mean(log_pz + log_jacob)

scales = np.linspace(0.3, 4.0, 100)
shifts_try = [-2.0, 0.0, 2.0]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for sh, color in zip(shifts_try, ['steelblue', 'darkorange', 'crimson']):
    lls = [log_likelihood_affine(x_data, sc, sh) for sc in scales]
    axes[0].plot(scales, lls, color=color, lw=2, label=f"shift={sh}")

axes[0].set_title("Affine Flow: Log-Likelihood vs scale\n(different shift values)", fontsize=11)
axes[0].set_xlabel("scale (sigma)"); axes[0].set_ylabel("Mean Log-Likelihood")
axes[0].legend(); axes[0].grid(alpha=0.3)
axes[0].axvline(0.5, color='gray', linestyle='--', lw=1, label="True std~0.5")

# Base vs transformed sampling
z_samples = np.random.randn(1000)
x_samples = affine_flow(z_samples, 0.5 * 2, 0.0)

z_plot = np.linspace(-5, 5, 300)
axes[1].hist(x_data, bins=40, density=True, alpha=0.5, color='gray', label="Real data")
axes[1].hist(x_samples, bins=40, density=True, alpha=0.5, color='steelblue', label="Flow samples")
axes[1].set_title("Data vs Affine Flow Samples", fontsize=11)
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig("fig_nb12_loglik.png", dpi=100, bbox_inches='tight')
plt.show()


## 3. Deep Flow: Layer Chain (Ch. 16.2)

A deep flow is a composition of K layers:

$$x = f_K(f_{K-1}(\\cdots f_1(z) \\cdots))$$

The Jacobian determinant is computed via the **product rule**:

$$\\log\\left|\\frac{\\partial f}{\\partial z}\\right| = \\sum_{k=1}^K \\log\\left|\\frac{\\partial f_k}{\\partial f_{k-1}}\\right|$$

This allows each layer's log-Jacobian to be computed and summed separately.


In [ ]:
# Deep flow: each layer is an affine transformation (demo)
class AffineLayer:
    def __init__(self, log_scale, shift):
        self.log_scale = log_scale
        self.shift = shift

    def forward(self, z):
        scale = np.exp(self.log_scale)
        return scale * z + self.shift, self.log_scale   # (x, log|J|)

    def inverse(self, x):
        scale = np.exp(self.log_scale)
        return (x - self.shift) / scale

class DeepFlow:
    def __init__(self, layers):
        self.layers = layers

    def forward(self, z):
        x = z.copy()
        log_det_total = 0.0
        for layer in self.layers:
            x, ldj = layer.forward(x)
            log_det_total += ldj
        return x, log_det_total

    def inverse(self, x):
        z = x.copy()
        for layer in reversed(self.layers):
            z = layer.inverse(z)
        return z

    def log_likelihood(self, x_data):
        z = self.inverse(x_data)
        log_pz = scipy_norm.logpdf(z)
        _, ldj = self.forward(z)
        return np.mean(log_pz - ldj)

# Richer transformation as number of layers increases
np.random.seed(7)
z_base = np.random.randn(2000)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
z_plot_hist = np.linspace(-5, 5, 300)

axes[0].hist(z_base, bins=50, density=True, color='steelblue', alpha=0.8, edgecolor='white')
axes[0].plot(z_plot_hist, scipy_norm.pdf(z_plot_hist), 'r--', lw=2)
axes[0].set_title("Base: z ~ N(0,1)", fontsize=10)
axes[0].set_xlim(-5, 5)

for ax, n_layers, color in zip(axes[1:], [1, 3, 6],
                                ['darkorange', 'crimson', 'purple']):
    layers = [AffineLayer(
                log_scale=np.random.randn() * 0.3,
                shift=np.random.randn() * 0.5)
              for _ in range(n_layers)]
    flow = DeepFlow(layers)
    x_transformed, _ = flow.forward(z_base)
    ll = flow.log_likelihood(x_transformed)

    ax.hist(x_transformed, bins=50, density=True, color=color, alpha=0.8, edgecolor='white')
    ax.set_title(f"{n_layers} Affine Layers\nlog-lik={ll:.2f}", fontsize=10)
    ax.set_xlim(-10, 10)

plt.suptitle("Deep Flow: Richer Transformation with More Layers", fontsize=12)
plt.tight_layout()
plt.savefig("fig_nb12_deep_flow.png", dpi=100, bbox_inches='tight')
plt.show()


## 4. Elementwise Flow: Leaky ReLU and Affine (Ch. 16.3)

The simplest invertible layer — transforms each dimension **independently**:

$$h'_d = g(h_d, \\phi_d)$$

**Leaky ReLU** is invertible because it is monotone:

$$\\text{LeakyReLU}(z) = \\begin{cases} z & z \\geq 0 \\\\ 0.1z & z < 0 \\end{cases}, \\quad \\text{Inverse: } z = \\begin{cases} x & x \\geq 0 \\\\ x/0.1 & x < 0 \\end{cases}$$

**Jacobian determinant** for elementwise flows is diagonal → O(D) computation.


In [ ]:
# Invertible activation functions
z_arr = np.linspace(-3, 3, 500)

def leaky_relu(z, alpha=0.1):
    return np.where(z >= 0, z, alpha * z)

def leaky_relu_inv(x, alpha=0.1):
    return np.where(x >= 0, x, x / alpha)

def leaky_relu_logJ(z, alpha=0.1):
    return np.where(z >= 0, 0.0, np.log(alpha))  # log|df/dz|

# Piecewise linear flow (Figure 16.5 in the textbook)
def piecewise_linear(h, phi):
    K = len(phi)
    phi = np.array(phi) / sum(phi)   # normalize
    phi_cumsum = np.concatenate([[0], np.cumsum(phi)])
    b = int(min(np.floor(h * K), K-1))
    return phi_cumsum[b] + (h * K - b) * phi[b]

piecewise_linear_v = np.vectorize(piecewise_linear, excluded=['phi'])

phi_params = [0.5, 0.3, 1.2, 0.8, 0.7]  # positive parameters
h_range = np.linspace(0, 1, 300)
pw_out = piecewise_linear_v(h_range, phi=phi_params)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].plot(z_arr, leaky_relu(z_arr), 'steelblue', lw=2.5, label="Leaky ReLU")
axes[0].plot(z_arr, z_arr, 'gray', lw=1, linestyle='--', label="Identity (y=z)")
axes[0].set_title("Leaky ReLU\n(Invertible Activation)", fontsize=11)
axes[0].set_xlabel("z"); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(z_arr, leaky_relu_logJ(z_arr), 'crimson', lw=2.5)
axes[1].axhline(0, color='gray', lw=1, linestyle='--', label="z>=0 region")
axes[1].set_title("Leaky ReLU Log-Jacobian\nlog|df/dz|", fontsize=11)
axes[1].set_xlabel("z"); axes[1].set_ylabel("log|J|"); axes[1].grid(alpha=0.3)
axes[1].legend()

axes[2].plot(h_range, pw_out, 'darkorange', lw=2.5)
axes[2].plot(h_range, h_range, 'gray', lw=1, linestyle='--')
axes[2].set_title("Piecewise Linear Flow\n(Figure 16.5, K=5 bins)", fontsize=11)
axes[2].set_xlabel("h"); axes[2].set_ylabel("h'"); axes[2].grid(alpha=0.3)

plt.suptitle("Elementwise Invertible Functions", fontsize=13)
plt.tight_layout()
plt.savefig("fig_nb12_elementwise.png", dpi=100, bbox_inches='tight')
plt.show()

# Log-Jacobian verification
z_test = np.array([-2.0, -1.0, 0.0, 1.0, 2.0])
print("Leaky ReLU log-Jacobian check:")
for z in z_test:
    lj = leaky_relu_logJ(z)
    print(f"  z={z:5.1f} -> log|J|={lj:.3f}  (|J|={np.exp(lj):.3f})")


## 5. Coupling Flow (Ch. 16.3.2)

The most common architecture in practice. The input is split into two parts:

$$h' = (h_1, \\; h_2'), \\quad h_1' = h_1, \\quad h_2' = g(h_2, \\; \\phi[h_1])$$

- $h_1$ passes through **unchanged** → simplifies the inverse computation via $h_1$
- $h_2$ undergoes a **transformation** parameterized by $\\phi[h_1]$
- $g$: elementwise invertible function (affine, spline…)

**Jacobian:** Lower triangular → determinant is O(D)!

Models such as RealNVP and GLOW are based on this idea.


In [ ]:
class AffineCouplingLayer:
    def __init__(self, d_in, split_idx, scale_net_w, translate_net_w):
        self.split_idx = split_idx
        self.sw = scale_net_w       # (d2, d1) weight
        self.tw = translate_net_w   # (d2, d1) weight

    def _scale_translate(self, h1):
        # Simple 1-layer network: tanh activation
        log_s = np.tanh(h1 @ self.sw.T)     # (N, d2)
        t     = np.tanh(h1 @ self.tw.T)     # (N, d2)
        return log_s, t

    def forward(self, h):
        h1 = h[:, :self.split_idx]
        h2 = h[:, self.split_idx:]
        log_s, t = self._scale_translate(h1)
        h2_prime = h2 * np.exp(log_s) + t
        log_det = log_s.sum(axis=1)     # elementwise → sum
        return np.concatenate([h1, h2_prime], axis=1), log_det

    def inverse(self, x):
        h1 = x[:, :self.split_idx]
        x2 = x[:, self.split_idx:]
        log_s, t = self._scale_translate(h1)
        h2 = (x2 - t) * np.exp(-log_s)
        return np.concatenate([h1, h2], axis=1)

# Coupling flow on 2D data
np.random.seed(3)
N = 500; d = 2; split = 1

sw = np.random.randn(1, 1) * 0.5
tw = np.random.randn(1, 1) * 0.5
layer = AffineCouplingLayer(d, split, sw, tw)

# Sample from Gaussian and transform
z_2d = np.random.randn(N, 2)
x_2d, ldj = layer.forward(z_2d)
z_rec = layer.inverse(x_2d)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
lim = 4

axes[0].scatter(z_2d[:,0], z_2d[:,1], alpha=0.3, s=8, color='steelblue')
axes[0].set_title("Base z ~ N(0,I)", fontsize=11)
axes[0].set_xlim(-lim, lim); axes[0].set_ylim(-lim, lim)
axes[0].set_xlabel("z1"); axes[0].set_ylabel("z2"); axes[0].grid(alpha=0.3)

axes[1].scatter(x_2d[:,0], x_2d[:,1], alpha=0.3, s=8, color='crimson')
axes[1].set_title("Transformed x = f(z)\n(Coupling layer)", fontsize=11)
axes[1].set_xlim(-lim, lim); axes[1].set_ylim(-lim, lim)
axes[1].set_xlabel("x1"); axes[1].set_ylabel("x2"); axes[1].grid(alpha=0.3)

recon_err = np.max(np.abs(z_rec - z_2d))
axes[2].scatter(z_rec[:,0], z_rec[:,1], alpha=0.3, s=8, color='limegreen')
axes[2].set_title(f"Inverse z = f⁻¹(x)\nMax error: {recon_err:.2e}", fontsize=11)
axes[2].set_xlim(-lim, lim); axes[2].set_ylim(-lim, lim)
axes[2].set_xlabel("z1 (recovered)"); axes[2].set_ylabel("z2 (recovered)")
axes[2].grid(alpha=0.3)

plt.suptitle("Affine Coupling Layer: Forward / Inverse Transform", fontsize=13)
plt.tight_layout()
plt.savefig("fig_nb12_coupling.png", dpi=100, bbox_inches='tight')
plt.show()

print(f"Coupling flow inverse max error: {recon_err:.2e}")
print(f"Log-determinant sample (first 5): {ldj[:5].round(3)}")


## 6. Autoregressive Flow (Ch. 16.3.3)

A generalization of coupling flow — each dimension depends on all preceding dimensions:

$$h'_d = g(h_d, \\phi_d[h_{1:d-1}])$$

- **Forward direction** (sampling): Parallel computation → fast
- **Inverse direction** (density evaluation): Sequential computation → slow

**Masked Autoregressive Flow (MAF):** Masks connections similar to the causal mask in Transformers.


In [ ]:
# Autoregressive flow — conceptual simulation
def autoregressive_forward(z, get_params_fn):
    '''
    z: (N, D) - base samples
    get_params_fn(h_prev): returns (log_scale, shift)
    '''
    D = z.shape[1]
    h = np.zeros_like(z)
    log_det = np.zeros(z.shape[0])

    for d in range(D):
        h_prev = h[:, :d]    # preceding dimensions
        log_s, t = get_params_fn(h_prev, d)
        h[:, d] = z[:, d] * np.exp(log_s) + t
        log_det += log_s

    return h, log_det

def get_params_simple(h_prev, d):
    # Simple autoregressive parameters: mean of preceding dimensions
    if d == 0:
        log_s = np.zeros(h_prev.shape[0] if h_prev.ndim > 1 else 1)
        t     = np.zeros_like(log_s)
    else:
        log_s = np.tanh(h_prev.mean(axis=1)) * 0.5
        t     = h_prev.mean(axis=1) * 0.3
    return log_s, t

np.random.seed(9)
z_ar = np.random.randn(1000, 4)
h_ar, ldj_ar = autoregressive_forward(z_ar, get_params_simple)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Autoregressive dependency structure
D_ar = 4
im = np.zeros((D_ar, D_ar))
for i in range(D_ar):
    for j in range(D_ar):
        if j < i: im[i, j] = 1   # h'_i depends on h_j (j < i)

axes[0].imshow(im, cmap='Blues', vmin=0, vmax=1)
axes[0].set_title("Autoregressive Dependency Structure\n(Row=output, Column=input)", fontsize=11)
axes[0].set_xticks(range(D_ar)); axes[0].set_yticks(range(D_ar))
axes[0].set_xticklabels([f"h{d}" for d in range(D_ar)])
axes[0].set_yticklabels([f"h'{d}" for d in range(D_ar)])
for i in range(D_ar):
    for j in range(D_ar):
        txt = "dep" if im[i,j] else ("free" if i==j else "-")
        axes[0].text(j, i, txt, ha='center', va='center', fontsize=7,
                     color='white' if im[i,j] else 'gray')

# Marginal distributions of each dimension
for d in range(D_ar):
    axes[1].hist(h_ar[:, d], bins=40, density=True, alpha=0.5, label=f"h'{d}")
axes[1].set_title("Autoregressive Flow Outputs\n(Marginal distribution per dimension)", fontsize=11)
axes[1].set_xlabel("Value"); axes[1].legend(fontsize=9); axes[1].grid(alpha=0.3)

plt.suptitle("Autoregressive Flow: Dependency Structure and Output Distribution", fontsize=13)
plt.tight_layout()
plt.savefig("fig_nb12_autoregressive.png", dpi=100, bbox_inches='tight')
plt.show()

print(f"Log-det statistics: mean={ldj_ar.mean():.3f}, std={ldj_ar.std():.3f}")


## 7. Real Training: 2D Moon-Shaped Distribution (Ch. 16.1.3)

Let us train a normalizing flow on real data.

**Target:** Learn the make_moons distribution  
**Model:** A few affine coupling layers  
**Loss:** Negative log-likelihood = $-\\log p_z(z) + \\log|J|$


In [ ]:
class RealNVP2D:
    '''
    Simple RealNVP for 2D: multiple Affine Coupling layers.
    The roles of x1 and x2 are swapped at alternating layers.
    '''
    def __init__(self, n_layers=4, hidden=8):
        self.n_layers = n_layers
        self.params = []
        np.random.seed(12)
        for k in range(n_layers):
            Ws1 = np.random.randn(hidden, 1) * 0.1
            bs1 = np.zeros(hidden)
            Ws2 = np.random.randn(1, hidden) * 0.1
            bs2 = np.zeros(1)
            Wt1 = np.random.randn(hidden, 1) * 0.1
            bt1 = np.zeros(hidden)
            Wt2 = np.random.randn(1, hidden) * 0.1
            bt2 = np.zeros(1)
            self.params.append({'Ws1':Ws1,'bs1':bs1,'Ws2':Ws2,'bs2':bs2,
                                 'Wt1':Wt1,'bt1':bt1,'Wt2':Wt2,'bt2':bt2,
                                 'flip': k % 2 == 1})

    def _net(self, x, W1, b1, W2, b2):
        return (x @ W1.T + b1) @ W2.T + b2   # linear output

    def _scale_translate(self, x1, p):
        log_s = np.tanh(self._net(x1, p['Ws1'],p['bs1'],p['Ws2'],p['bs2']))
        t     = self._net(x1, p['Wt1'],p['bt1'],p['Wt2'],p['bt2'])
        return log_s.flatten(), t.flatten()

    def forward(self, z):
        x = z.copy(); log_det = np.zeros(len(z))
        for p in self.params:
            idx1, idx2 = (0, 1) if not p['flip'] else (1, 0)
            x1 = x[:, idx1:idx1+1]
            x2 = x[:, idx2:idx2+1]
            log_s, t = self._scale_translate(x1, p)
            x2_new = x2.flatten() * np.exp(log_s) + t
            x_new = np.zeros_like(x)
            x_new[:, idx1] = x1.flatten()
            x_new[:, idx2] = x2_new
            x = x_new
            log_det += log_s
        return x, log_det

    def inverse(self, x_in):
        x = x_in.copy()
        for p in reversed(self.params):
            idx1, idx2 = (0, 1) if not p['flip'] else (1, 0)
            x1 = x[:, idx1:idx1+1]
            x2 = x[:, idx2:idx2+1]
            log_s, t = self._scale_translate(x1, p)
            x2_orig = (x2.flatten() - t) * np.exp(-log_s)
            x_new = np.zeros_like(x)
            x_new[:, idx1] = x1.flatten()
            x_new[:, idx2] = x2_orig
            x = x_new
        return x

    def log_prob(self, x_data):
        _, ldj = self.forward(self.inverse(x_data))
        z_inv = self.inverse(x_data)
        log_pz = scipy_norm.logpdf(z_inv).sum(axis=1)
        return log_pz + ldj

# Prepare data
np.random.seed(0)
X_moons, _ = make_moons(n_samples=1000, noise=0.1)
X_moons = (X_moons - X_moons.mean(0)) / X_moons.std(0)  # normalize

model = RealNVP2D(n_layers=6, hidden=8)

# Short training with numerical gradients (for illustration)
lr = 0.002
losses = []
all_params = []
for p in model.params:
    for key in ['Ws1','bs1','Ws2','bs2','Wt1','bt1','Wt2','bt2']:
        all_params.append((p, key))

for epoch in range(80):
    idx = np.random.choice(len(X_moons), 64, replace=False)
    batch = X_moons[idx]
    lp = model.log_prob(batch)
    loss = -lp.mean()
    losses.append(loss)

    eps_num = 1e-3
    for p, key in all_params:
        orig = p[key].copy()
        grad = np.zeros_like(orig)
        for idx_flat in np.ndindex(orig.shape):
            p[key][idx_flat] += eps_num
            lp_plus = -model.log_prob(batch).mean()
            p[key][idx_flat] -= 2*eps_num
            lp_minus = -model.log_prob(batch).mean()
            p[key][idx_flat] = orig[idx_flat]
            grad[idx_flat] = (lp_plus - lp_minus) / (2*eps_num)
        p[key] -= lr * np.clip(grad, -1, 1)

print(f"Initial loss: {losses[0]:.3f}")
print(f"Final loss:   {losses[-1]:.3f}")

# Visualize
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(losses, color='steelblue', lw=2)
axes[0].set_title("Training Loss (NLL)", fontsize=11)
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("-log p(x)")
axes[0].grid(alpha=0.3)

z_new = np.random.randn(500, 2)
x_gen, _ = model.forward(z_new)
axes[1].scatter(X_moons[:,0], X_moons[:,1], s=8, alpha=0.4, color='steelblue', label="Real")
axes[1].scatter(x_gen[:,0], x_gen[:,1], s=8, alpha=0.4, color='crimson', label="Flow sample")
axes[1].set_title("Real Data vs Flow Samples", fontsize=11)
axes[1].legend(fontsize=9); axes[1].grid(alpha=0.3)
axes[1].set_xlim(-3.5, 3.5); axes[1].set_ylim(-3.5, 3.5)

xx, yy = np.meshgrid(np.linspace(-3,3,50), np.linspace(-3,3,50))
grid = np.c_[xx.ravel(), yy.ravel()]
log_probs = model.log_prob(grid)
log_probs_grid = log_probs.reshape(50, 50)
axes[2].contourf(xx, yy, np.exp(np.clip(log_probs_grid, -10, 0)),
                  levels=20, cmap='YlOrRd', alpha=0.9)
axes[2].scatter(X_moons[:,0], X_moons[:,1], s=5, alpha=0.3, color='black')
axes[2].set_title("Learned Density Map", fontsize=11)
axes[2].set_xlim(-3, 3); axes[2].set_ylim(-3, 3)

plt.suptitle("RealNVP-Like Flow: Learning the make_moons Distribution", fontsize=13)
plt.tight_layout()
plt.savefig("fig_nb12_training.png", dpi=100, bbox_inches='tight')
plt.show()


## 8. Generative Model Comparison (Ch. 16.6)

| Feature | GAN | VAE | **Normalizing Flow** | Diffusion |
|---|---|---|---|---|
| Training | Adversarial | ELBO max. | **NLL min. (exact MLE)** | Noise prediction |
| Density $p(x)$ | ❌ None | ≈ Lower bound | **✅ Exact** | ≈ Lower bound |
| Sampling | ✅ Fast | ✅ Fast | **✅ Fast** | ❌ Slow (T steps) |
| Invertibility | ❌ | ❌ | **✅ Exact** | ❌ |
| Memory | Low | Low | **High** (inv. network) | Medium |
| Image quality | High | Medium | Medium | Highest |

**The unique advantage of Normalizing Flows:** Exact computation of $p(x)$ →  
Ideal for anomaly detection, lossless compression, and Bayesian inference.


In [ ]:
# Anomaly detection example: flow uses exact log p(x) knowledge
np.random.seed(4)

# 'Normal' data: tight Gaussian
normal_data = np.random.randn(300, 2) * 0.5

# 'Anomaly' data: far-away points
anomaly_data = np.random.randn(30, 2) * 0.5 + np.array([[4.0, 0.0]])

all_data = np.vstack([normal_data, anomaly_data])

# Simple flow: learn mean and variance (MVN fit)
mu_fit = normal_data.mean(axis=0)
cov_fit = np.cov(normal_data.T)
cov_inv = np.linalg.inv(cov_fit)
log_det_cov = np.log(np.linalg.det(cov_fit))

def mvn_log_prob(x, mu, cov_inv, log_det):
    d = x - mu
    maha = np.sum(d @ cov_inv * d, axis=1)
    return -0.5 * (maha + log_det + 2 * np.log(2 * np.pi))

log_probs_all = mvn_log_prob(all_data, mu_fit, cov_inv, log_det_cov)
threshold = np.percentile(log_probs_all[:300], 5)  # 5th percentile threshold

is_anomaly_pred = log_probs_all < threshold

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

scatter = axes[0].scatter(all_data[:,0], all_data[:,1],
                           c=log_probs_all, cmap='RdYlGn',
                           s=30, alpha=0.8, vmin=log_probs_all.min(),
                           vmax=log_probs_all.max())
plt.colorbar(scatter, ax=axes[0], label="log p(x)")
axes[0].set_title("Anomaly Detection with Normalizing Flow\nColor = log p(x)", fontsize=11)
axes[0].set_xlabel("x1"); axes[0].set_ylabel("x2")
axes[0].grid(alpha=0.3)

# Classification with threshold
colors_pred = ['crimson' if a else 'steelblue' for a in is_anomaly_pred]
axes[1].scatter(all_data[:,0], all_data[:,1], c=colors_pred, s=30, alpha=0.8)
axes[1].set_title(f"Classification with threshold (log p < {threshold:.1f})\n"
                   f"Detected: {is_anomaly_pred.sum()} anomalies", fontsize=11)
axes[1].set_xlabel("x1")

legend_elems = [plt.scatter([], [], color='crimson',   s=40, label='Anomaly (predicted)'),
                plt.scatter([], [], color='steelblue', s=40, label='Normal (predicted)')]
axes[1].legend(handles=legend_elems, fontsize=9)
axes[1].grid(alpha=0.3)

plt.suptitle("Anomaly Detection with Flow Model (log p(x) < threshold → anomaly)", fontsize=12)
plt.tight_layout()
plt.savefig("fig_nb12_anomaly.png", dpi=100, bbox_inches='tight')
plt.show()

n_correct = is_anomaly_pred[300:].sum()
print(f"True anomalies detected: {n_correct}/30 ({100*n_correct/30:.0f}%)")


## 9. Summary

| Concept | Description |
|---|---|
| **Normalizing Flow** | Transform simple → complex distribution via an invertible network |
| **Change of variables** | $p_x(x) = p_z(f^{-1}(x)) \\cdot \\|J^{-1}\\|$ |
| **Jacobian determinant** | Measures volume change; added to log-likelihood |
| **Elementwise flow** | Transform each dimension independently; log-J diagonal → O(D) |
| **Coupling flow** | Split input in two; transform one part conditioned on the other |
| **Autoregressive flow** | Chain of dependencies; parallel sampling, sequential inverse |
| **Exact MLE** | The distinguishing property of flows: exact computation of $p(x)$ |
| **Anomaly detection** | Low $\\log p(x)$ → anomalous |
| **RealNVP / GLOW** | Image generation models based on coupling flows |
